# ForexFactory Scraper for Economic Event Surprises

**Project:** Economic Event-Driven Forex Trading Assistant (FYP)

**Purpose:** Scrape the ForexFactory economic calendar to extract analyst consensus forecasts and actual released values for both USD and EUR macroeconomic events. The difference (actual minus forecast) provides the market surprise used as a feature in the main ML pipeline.

**Coverage:** January 2007 to December 2025.

**Events captured:**
- **USD:** CPI m/m, Non-Farm Employment Change (NFP), Federal Funds Rate (FOMC)
- **EUR:** Main Refinancing Rate (ECB), CPI Flash Estimate y/y, Core CPI Flash Estimate y/y

**Output:** Six CSV files, each containing two columns (`date`, `<event>_surprise`):
- `cpi_surprise.csv`, `nfp_surprise.csv`, `fomc_surprise.csv`
- `ecb_rate_surprise.csv`, `eu_cpi_surprise.csv`, `eu_core_cpi_surprise.csv`

**Technical note:** ForexFactory uses Cloudflare bot protection. The `cloudscraper` library bypasses this by emulating a browser fingerprint, allowing programmatic access without requiring API authentication.

In [1]:
# install dependencies (run once, then comment out)
# %pip install cloudscraper beautifulsoup4 pandas

import cloudscraper
import pandas as pd
from bs4 import BeautifulSoup
import time

# initialise cloudscraper session (handles Cloudflare bypass)
scraper = cloudscraper.create_scraper()

print("Setup complete")

Setup complete


## Section 1: Scraper Function

A single parameterised scraper function handles both USD and EUR events. The function takes:
- `year` and `month`: which calendar month to scrape
- `currency`: filter to a specific currency code (e.g. 'USD' or 'EUR')
- `event_keywords`: list of event name substrings to keep

This design avoids code duplication and makes it trivial to extend to additional currencies (e.g. GBP from the Bank of England) in future work.

In [2]:
def scrape_forexfactory(year, month, currency, event_keywords):
    """
    Scrape ForexFactory economic calendar for one month.

    Parameters
    ----------
    year : int
        Year to scrape (e.g. 2024).
    month : int
        Month number (1-12).
    currency : str
        ISO currency code to filter on (e.g. 'USD', 'EUR').
    event_keywords : list of str
        Event name substrings to keep (e.g. ['CPI', 'Non-Farm', 'Fed']).
        A row is kept if any keyword appears in the event name.

    Returns
    -------
    pandas.DataFrame
        Columns: date, year, month, currency, event, actual, forecast, previous.
    """
    month_names = ['jan', 'feb', 'mar', 'apr', 'may', 'jun',
                   'jul', 'aug', 'sep', 'oct', 'nov', 'dec']
    url = f"https://www.forexfactory.com/calendar?month={month_names[month-1]}.{year}"

    response = scraper.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    rows = soup.find_all('tr', class_='calendar__row')

    results = []
    current_date = None

    for row in rows:
        # ForexFactory only shows the date on the first row of each day
        date_cell = row.find('td', class_='calendar__date')
        if date_cell and date_cell.text.strip():
            current_date = date_cell.text.strip()

        event_cell = row.find('td', class_='calendar__event')
        if not event_cell:
            continue
        event_name = event_cell.text.strip()

        # currency filter
        currency_cell = row.find('td', class_='calendar__currency')
        if not currency_cell or currency_cell.text.strip() != currency:
            continue

        # event keyword filter
        if not any(k in event_name for k in event_keywords):
            continue

        actual = row.find('td', class_='calendar__actual')
        forecast = row.find('td', class_='calendar__forecast')
        previous = row.find('td', class_='calendar__previous')

        results.append({
            'date':     current_date,
            'year':     year,
            'month':    month,
            'currency': currency,
            'event':    event_name,
            'actual':   actual.text.strip()   if actual   else '',
            'forecast': forecast.text.strip() if forecast else '',
            'previous': previous.text.strip() if previous else '',
        })

    return pd.DataFrame(results)


# quick smoke test on one month
test_df = scrape_forexfactory(2024, 1, 'USD', ['CPI', 'Non-Farm', 'Fed'])
print(f"Smoke test (Jan 2024 USD): {len(test_df)} rows found")
test_df.head()

Smoke test (Jan 2024 USD): 9 rows found


,date,year,month,currency,event,actual,forecast,previous
0,Thu Jan 4,2024,1,USD,ADP Non-Farm Employment Change,164K,120K,101K
1,Fri Jan 5,2024,1,USD,Non-Farm Employment Change,216K,168K,173K
2,Thu Jan 11,2024,1,USD,Core CPI m/m,0.3%,0.3%,0.3%
3,Thu Jan 11,2024,1,USD,Core CPI y/y,3.9%,3.8%,4.0%
4,Thu Jan 11,2024,1,USD,CPI m/m,0.3%,0.2%,0.1%


## Section 2: Run the Scrape

Loop through every month from January 2019 to December 2025 for both currencies. The `time.sleep(3)` between requests avoids triggering ForexFactory's rate-limit / Cloudflare re-challenge.

Expected runtime: approximately 4-5 minutes per currency (84 months × 3 seconds).

In [3]:
# USD events: CPI, Non-Farm Payrolls, Federal Funds Rate
usd_keywords = ['CPI', 'Non-Farm', 'NFP', 'Fed', 'Interest Rate']
usd_data = []

for year in range(2007, 2026):
    for month in range(1, 13):
        print(f"USD {year}-{month:02d}...", end=' ')
        try:
            df_m = scrape_forexfactory(year, month, 'USD', usd_keywords)
            usd_data.append(df_m)
            print(f"{len(df_m)} events")
            time.sleep(3)
        except Exception as e:
            print(f"FAILED: {e}")
            continue

usd_raw = pd.concat(usd_data, ignore_index=True)
print(f"\nTotal USD events scraped: {len(usd_raw)}")
print(usd_raw['event'].value_counts())

USD 2007-01... 11 events
USD 2007-02... 13 events
USD 2007-03... 14 events
USD 2007-04... 10 events
USD 2007-05... 14 events
USD 2007-06... 11 events
USD 2007-07... 13 events
USD 2007-08... 11 events
USD 2007-09... 13 events
USD 2007-10... 14 events
USD 2007-11... 13 events
USD 2007-12... 10 events
USD 2008-01... 13 events
USD 2008-02... 11 events
USD 2008-03... 14 events
USD 2008-04... 14 events
USD 2008-05... 15 events
USD 2008-06... 14 events
USD 2008-07... 15 events
USD 2008-08... 9 events
USD 2008-09... 17 events
USD 2008-10... 19 events
USD 2008-11... 13 events
USD 2008-12... 11 events
USD 2009-01... 11 events
USD 2009-02... 15 events
USD 2009-03... 18 events
USD 2009-04... 14 events
USD 2009-05... 15 events
USD 2009-06... 16 events
USD 2009-07... 17 events
USD 2009-08... 11 events
USD 2009-09... 13 events
USD 2009-10... 13 events
USD 2009-11... 10 events
USD 2009-12... 11 events
USD 2010-01... 11 events
USD 2010-02... 14 events
USD 2010-03... 14 events
USD 2010-04... 17 events
U

In [4]:
# EUR events: ECB Main Refinancing Rate, CPI Flash Estimate (regular and Core)
eur_keywords = ['Main Refinancing Rate', 'CPI Flash Estimate']
eur_data = []

for year in range(2007, 2026):
    for month in range(1, 13):
        print(f"EUR {year}-{month:02d}...", end=' ')
        try:
            df_m = scrape_forexfactory(year, month, 'EUR', eur_keywords)
            eur_data.append(df_m)
            print(f"{len(df_m)} events")
            time.sleep(3)
        except Exception as e:
            print(f"FAILED: {e}")
            continue

eur_raw = pd.concat(eur_data, ignore_index=True)
print(f"\nTotal EUR events scraped: {len(eur_raw)}")
print(eur_raw['event'].value_counts())

EUR 2007-01... 3 events
EUR 2007-02... 1 events
EUR 2007-03... 3 events
EUR 2007-04... 2 events
EUR 2007-05... 2 events
EUR 2007-06... 2 events
EUR 2007-07... 2 events
EUR 2007-08... 2 events
EUR 2007-09... 2 events
EUR 2007-10... 2 events
EUR 2007-11... 2 events
EUR 2007-12... 1 events
EUR 2008-01... 3 events
EUR 2008-02... 1 events
EUR 2008-03... 3 events
EUR 2008-04... 2 events
EUR 2008-05... 2 events
EUR 2008-06... 2 events
EUR 2008-07... 2 events
EUR 2008-08... 2 events
EUR 2008-09... 2 events
EUR 2008-10... 3 events
EUR 2008-11... 2 events
EUR 2008-12... 1 events
EUR 2009-01... 3 events
EUR 2009-02... 1 events
EUR 2009-03... 3 events
EUR 2009-04... 2 events
EUR 2009-05... 2 events
EUR 2009-06... 2 events
EUR 2009-07... 2 events
EUR 2009-08... 2 events
EUR 2009-09... 2 events
EUR 2009-10... 2 events
EUR 2009-11... 2 events
EUR 2009-12... 1 events
EUR 2010-01... 3 events
EUR 2010-02... 1 events
EUR 2010-03... 3 events
EUR 2010-04... 2 events
EUR 2010-05... 2 events
EUR 2010-06... 2

## Section 3: Data Cleaning

Two cleaning steps:

1. **Number cleaning**: ForexFactory formats values with suffixes (`%`, `K`, `M`) and qualifiers (`<`). Strip these and convert to float. Compute `surprise = actual - forecast`.

2. **Date parsing**: ForexFactory's date format is `Day Mon DD` (e.g. `Fri Jan 12`) without a year on the page. Combine with the scraped year, strip the weekday prefix, and parse to a proper datetime.

In [5]:
def clean_numbers(df):
    """
    Clean ForexFactory numeric columns and compute surprise.

    Strips %, K, M, < symbols from actual / forecast / previous columns,
    converts to float (NaN on failure), and adds a `surprise` column
    computed as actual minus forecast.
    """
    df = df.copy()
    for col in ['actual', 'forecast', 'previous']:
        if col not in df.columns:
            continue
        df[col] = (
            df[col].astype(str)
                   .str.replace('%', '', regex=False)
                   .str.replace('K', '', regex=False)
                   .str.replace('M', '', regex=False)
                   .str.replace('<', '', regex=False)
                   .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df['surprise'] = df['actual'] - df['forecast']
    return df


def parse_dates(df):
    """
    Parse ForexFactory date strings into proper datetime values.

    Strips weekday prefix, appends year, handles missing space between
    month and day, and parses with the format 'Mon DD YYYY'.
    Rows with un-parseable dates are dropped.
    """
    df = df.copy()
    # strip weekday prefix (e.g. 'Fri Jan 12' -> 'Jan 12')
    df['date_clean'] = df['date'].astype(str).str.replace(
        r'(Mon|Tue|Wed|Thu|Fri|Sat|Sun)\s+', '', regex=True
    ).str.strip()
    # append the year
    df['date_full'] = df['date_clean'] + ' ' + df['year'].astype(str)
    # insert a space between text and digits (handles 'May23' edge case)
    df['date_full'] = df['date_full'].str.replace(
        r'([A-Za-z])([0-9])', r'\1 \2', regex=True
    )
    # collapse multiple spaces
    df['date_full'] = df['date_full'].str.replace(r'\s+', ' ', regex=True).str.strip()
    # parse to datetime
    df['date'] = pd.to_datetime(df['date_full'], format='%b %d %Y', errors='coerce')
    return df


# Apply number and date cleaning
usd_clean = parse_dates(clean_numbers(usd_raw))
eur_clean = parse_dates(clean_numbers(eur_raw))

# Remove only rows without a valid date
usd_clean = usd_clean.dropna(subset=['date']).copy()
eur_clean = eur_clean.dropna(subset=['date']).copy()

# Record whether actual-minus-forecast was available
usd_clean['surprise_available'] = (
    usd_clean['surprise'].notna().astype(int)
)

eur_clean['surprise_available'] = (
    eur_clean['surprise'].notna().astype(int)
)

In [6]:
print("USD event names:")
print(usd_clean['event'].value_counts().sort_index().to_string())

print("\nEUR event names:")
print(eur_clean['event'].value_counts().sort_index().to_string())

USD event names:
event
ADP Non-Farm Employment Change          226
CPI m/m                                 226
CPI y/y                                 227
Cleveland Fed Inflation Expectations     11
Core CPI m/m                            226
Core CPI y/y                            227
Fed Announcement                         53
Fed Chair Nomination Vote                 3
Fed Chair Yellen Speaks                  50
Fed Chair Yellen Testifies               24
Fed Chairman Bernanke Speaks            166
Fed Chairman Bernanke Testifies          73
Fed Chairman Powell Testifies            50
Fed Chairman Warsh Speaks                22
Fed Gov Nomination Hearings               9
Fed Monetary Policy Report               38
Federal Budget Balance                  228
Federal Funds Rate                      155
Non-Farm Employment Change              228
Philly Fed Manufacturing Index          228

EUR event names:
event
CPI Flash Estimate y/y         228
Core CPI Flash Estimate y/y    152
Mai

new step 2 auditing of event names

In [7]:
EVENT_NAME_MAP = {
    'CPI m/m': 'us_cpi',
    'Non-Farm Employment Change': 'us_nfp',
    'Federal Funds Rate': 'us_fomc',
    'Main Refinancing Rate': 'ecb_rate',
    'CPI Flash Estimate y/y': 'eu_cpi',
    'Core CPI Flash Estimate y/y': 'eu_core_cpi'
}

usd_clean['canonical_event'] = usd_clean['event'].map(EVENT_NAME_MAP)
eur_clean['canonical_event'] = eur_clean['event'].map(EVENT_NAME_MAP)

print(
    pd.concat([usd_clean, eur_clean])
      .loc[lambda x: x['canonical_event'].isna(), 'event']
      .value_counts()
)

event
Federal Budget Balance                  228
Philly Fed Manufacturing Index          228
Core CPI y/y                            227
CPI y/y                                 227
ADP Non-Farm Employment Change          226
Core CPI m/m                            226
Fed Chairman Bernanke Speaks            166
Fed Chairman Bernanke Testifies          73
Fed Announcement                         53
Fed Chair Yellen Speaks                  50
Fed Chairman Powell Testifies            50
Fed Monetary Policy Report               38
Fed Chair Yellen Testifies               24
Fed Chairman Warsh Speaks                22
Cleveland Fed Inflation Expectations     11
Fed Gov Nomination Hearings               9
Fed Chair Nomination Vote                 3
Name: count, dtype: int64


In [8]:
def prepare_event_data(
    source_df,
    event_names,
    surprise_col,
    available_col
):
    event_df = source_df[
        source_df['event'].isin(event_names)
    ][
        [
            'date',
            'event',
            'actual',
            'forecast',
            'previous',
            'surprise',
            'surprise_available'
        ]
    ].copy()

    event_df = event_df.sort_values('date')

    # Remove only exact duplicate releases
    event_df = event_df.drop_duplicates(
        subset=['date', 'event', 'actual', 'forecast'],
        keep='last'
    )

    event_df = event_df.rename(
        columns={
            'surprise': surprise_col,
            'surprise_available': available_col
        }
    )

    return event_df.reset_index(drop=True)

new step 3 removal true duplicate

In [9]:
all_events = pd.concat(
    [usd_clean, eur_clean],
    ignore_index=True
)

# Keep only the six events used by the project
all_events = all_events[
    all_events['canonical_event'].notna()
].copy()

In [10]:
all_events = all_events.drop_duplicates(
    subset=['date', 'canonical_event', 'actual', 'forecast'],
    keep='last'
)

new step 4 audit coverage by year

In [11]:
coverage = (
    all_events
    .dropna(subset=['canonical_event'])
    .assign(year=lambda x: x['date'].dt.year)
    .groupby(['year', 'canonical_event'])
    .size()
    .unstack(fill_value=0)
)

display(coverage)

canonical_event,ecb_rate,eu_core_cpi,eu_cpi,us_cpi,us_fomc,us_nfp
year,,,,,,
2007,12,0,12,12,8,12
2008,13,0,12,12,10,12
2009,12,0,12,12,8,12
2010,12,0,12,12,8,12
2011,12,0,12,12,8,12
2012,12,0,12,12,8,12
2013,12,8,12,12,8,12
2014,12,12,12,12,8,12
2015,8,12,12,12,8,12


new step 5 check missing surprises by event

In [12]:
missing_summary = (
    all_events
    .groupby('canonical_event')
    .agg(
        total_events=('date', 'size'),
        surprise_available=('surprise_available', 'sum')
    )
)

missing_summary['missing_surprise'] = (
    missing_summary['total_events']
    - missing_summary['surprise_available']
)

missing_summary['available_pct'] = (
    missing_summary['surprise_available']
    / missing_summary['total_events']
    * 100
)

display(missing_summary)

,total_events,surprise_available,missing_surprise,available_pct
canonical_event,,,,
ecb_rate,185,184,1,99.459459
eu_core_cpi,152,139,13,91.447368
eu_cpi,228,228,0,100.000000
us_cpi,226,226,0,100.000000
us_fomc,155,151,4,97.419355
us_nfp,228,227,1,99.561404


new step 6 check extreme nfp value

In [13]:
nfp_rows = all_events[
    all_events['canonical_event'] == 'us_nfp'
].copy()

display(
    nfp_rows.loc[
        nfp_rows['surprise'].abs().nlargest(10).index,
        ['date', 'actual', 'forecast', 'previous', 'surprise']
    ].sort_values('surprise')
)

,date,actual,forecast,previous,surprise
1944,2021-05-07,266.0,990.0,770.0,-724.0
1810,2020-04-03,-701.0,-100.0,275.0,-601.0
1982,2021-09-03,235.0,720.0,1053.0,-485.0
2012,2021-12-03,210.0,553.0,546.0,-343.0
1993,2021-10-08,194.0,490.0,366.0,-296.0
2141,2023-02-03,517.0,193.0,260.0,324.0
2032,2022-02-04,467.0,110.0,510.0,357.0
1821,2020-05-08,-20537.0,-22000.0,-881.0,1463.0
1844,2020-07-02,4800.0,3037.0,2699.0,1763.0
1830,2020-06-05,2509.0,-7750.0,-20687.0,10259.0


new step 7 save one master event file

In [14]:
master_events = all_events[
    [
        'date',
        'event',
        'canonical_event',
        'actual',
        'forecast',
        'previous',
        'surprise',
        'surprise_available'
    ]
].copy()

master_events = master_events.sort_values(
    ['date', 'canonical_event']
)

master_events.to_csv(
    'economic_events_master_2007_2025.csv',
    index=False
)

## Section 4: Filter and Save USD CSVs

Three USD CSVs corresponding to the three US economic releases used in the main pipeline. The event-name filter is strict (exact match on the canonical ForexFactory name) to avoid contamination from non-US events that share keywords (e.g. French CPI also contains 'CPI' in the name).

new

In [15]:
# Prepare USD event datasets
cpi_df = prepare_event_data(
    usd_clean,
    ['CPI m/m'],
    'cpi_surprise',
    'cpi_available'
)

nfp_df = prepare_event_data(
    usd_clean,
    ['Non-Farm Employment Change'],
    'nfp_surprise',
    'nfp_available'
)

fomc_df = prepare_event_data(
    usd_clean,
    ['Federal Funds Rate'],
    'fomc_surprise',
    'fomc_available'
)

# Save USD datasets
cpi_df.to_csv('cpi_surprise.csv', index=False)
nfp_df.to_csv('nfp_surprise.csv', index=False)
fomc_df.to_csv('fomc_surprise.csv', index=False)

print(f"USD CPI saved: {len(cpi_df)} events")
print(f"USD NFP saved: {len(nfp_df)} events")
print(f"USD FOMC saved: {len(fomc_df)} events")

USD CPI saved: 226 events
USD NFP saved: 228 events
USD FOMC saved: 155 events


## Section 5: Filter and Save ECB / EUR CSVs

Three EUR CSVs corresponding to the three Euro-area economic releases. Added in response to mid-review supervisor feedback recommending coverage of both sides of the EUR/USD pair.

In [16]:
# Prepare EUR event datasets
ecb_rate_df = prepare_event_data(
    eur_clean,
    ['Main Refinancing Rate'],
    'ecb_rate_surprise',
    'ecb_rate_available'
)

eu_cpi_df = prepare_event_data(
    eur_clean,
    ['CPI Flash Estimate y/y'],
    'eu_cpi_surprise',
    'eu_cpi_available'
)

eu_core_cpi_df = prepare_event_data(
    eur_clean,
    ['Core CPI Flash Estimate y/y'],
    'eu_core_cpi_surprise',
    'eu_core_cpi_available'
)

# Save EUR datasets
ecb_rate_df.to_csv(
    'ecb_rate_surprise.csv',
    index=False
)

eu_cpi_df.to_csv(
    'eu_cpi_surprise.csv',
    index=False
)

eu_core_cpi_df.to_csv(
    'eu_core_cpi_surprise.csv',
    index=False
)

print(f"ECB Rate saved: {len(ecb_rate_df)} events")
print(f"EU CPI saved: {len(eu_cpi_df)} events")
print(f"EU Core CPI saved: {len(eu_core_cpi_df)} events")

ECB Rate saved: 185 events
EU CPI saved: 228 events
EU Core CPI saved: 152 events


## Section 6: Sanity Checks

Distribution of surprise values for each event. Useful for verifying scraper output and for the methodology discussion in the final report.

Note: ECB Rate surprises are expected to be near-zero in most observations due to the ECB's forward guidance policy framework (decisions are telegraphed via speeches and meeting minutes weeks in advance, leaving little room for surprise on the announcement day). This is consistent with the macroeconomic announcement literature.

In [17]:
datasets = [
    ('USD CPI', cpi_df, 'cpi_surprise', 'cpi_available'),
    ('USD NFP', nfp_df, 'nfp_surprise', 'nfp_available'),
    ('USD FOMC', fomc_df, 'fomc_surprise', 'fomc_available'),
    (
        'ECB Rate',
        ecb_rate_df,
        'ecb_rate_surprise',
        'ecb_rate_available'
    ),
    (
        'EU CPI',
        eu_cpi_df,
        'eu_cpi_surprise',
        'eu_cpi_available'
    ),
    (
        'EU Core CPI',
        eu_core_cpi_df,
        'eu_core_cpi_surprise',
        'eu_core_cpi_available'
    ),
]

for name, event_df, surprise_col, available_col in datasets:
    valid_surprises = event_df.loc[
        event_df[available_col] == 1,
        surprise_col
    ]

    print(
        f"{name}: "
        f"total events={len(event_df)}, "
        f"surprises available={len(valid_surprises)}, "
        f"missing={len(event_df) - len(valid_surprises)}"
    )

    if not valid_surprises.empty:
        print(
            f"  mean={valid_surprises.mean():.3f}, "
            f"std={valid_surprises.std():.3f}, "
            f"min={valid_surprises.min():.3f}, "
            f"max={valid_surprises.max():.3f}, "
            f"non-zero={(valid_surprises != 0).sum()}"
        )

USD CPI: total events=226, surprises available=226, missing=0
  mean=0.000, std=0.133, min=-0.500, max=0.600, non-zero=148
USD NFP: total events=228, surprises available=227, missing=1
  mean=57.960, std=706.156, min=-724.000, max=10259.000, non-zero=225
USD FOMC: total events=155, surprises available=151, missing=4
  mean=-0.003, std=0.041, min=-0.250, max=0.250, non-zero=4
ECB Rate: total events=185, surprises available=184, missing=1
  mean=-0.001, std=0.046, min=-0.250, max=0.250, non-zero=9
EU CPI: total events=228, surprises available=228, missing=0
  mean=0.010, std=0.180, min=-0.500, max=0.800, non-zero=150
EU Core CPI: total events=152, surprises available=139, missing=13
  mean=0.014, std=0.144, min=-0.500, max=0.500, non-zero=92
